In [0]:
_checkpoints = "dbfs:/Volumes/workspace/retails/raw_data/_checkpoints/retails/dev/gold/products"

In [0]:
# -- # Read clean products from silver
silver_df = (
    spark.readStream \
        .format("delta") \
        .table("retails.silver.products_cdc")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, sha2, concat_ws

# Prepare dataframe for gold SCD2
silver_df = (
    silver_df
        .withColumn("product_price", col("product_price").cast("double")) \

        .select(
            "product_id",
            "product_name",
            "product_description",
            "product_price",
            "product_image",
            "product_category_id",
            "op"
        ) \

        .withColumn("is_active", lit(True).cast("boolean")) \
        .withColumn("effective_from", current_timestamp()) \
        .withColumn(
            "effective_to",
            lit(None).cast("timestamp")
        ) \
        .withColumn("is_current", lit(True).cast("boolean")) \
        .withColumn("created_ts", current_timestamp()) \
        .withColumn("updated_ts", current_timestamp()) \
        .withColumn("record_hash",
                        sha2(
                            concat_ws(
                                "||",
                                col("product_id"),
                                col("product_name"),
                                col("product_description"),
                                col("product_price"),
                                col("product_category_id"),
                                col("product_image")
                            ),
                            256
                        )
                    ))
    


In [0]:
# STEP-1: Expire old current rows
MERGE_EXPIRE = """
MERGE INTO retails.gold.dim_products t
USING silver_products_vw s

ON t.product_id = s.product_id
AND t.is_current = true

WHEN MATCHED AND s.op = 'UPDATE'
THEN UPDATE SET
    t.is_active = false,
    t.is_current = false,
    t.effective_to = current_timestamp(),
    t.updated_ts = current_timestamp()

WHEN MATCHED AND s.op = 'DELETE'
THEN UPDATE SET 
    t.is_active = false,
    t.is_current = false,
    t.effective_to = current_timestamp(),
    t.updated_ts = current_timestamp()
"""

In [0]:
# STEP-2: Insert new versions
INSERT_NEW = """
INSERT INTO retails.gold.dim_products(
    product_id,
    product_name,
    product_description,
    product_price,
    product_image,
    product_category_id,
    is_active,
    effective_from,
    effective_to,
    is_current,
    created_ts,
    updated_ts)

SELECT
    s.product_id,
    s.product_name,
    s.product_description,
    s.product_price,
    s.product_image,
    s.product_category_id,
    true,
    current_timestamp(),
    CAST(NULL AS TIMESTAMP),
    true,
    current_timestamp(),
    current_timestamp()

FROM silver_products_vw s

WHERE s.op IN ('INSERT', 'UPDATE')
"""

In [0]:
# foreachBatch function
def upsert_to_gold(batch_df, batch_id):

    batch_df.createOrReplaceTempView(
        "silver_products_vw"
    )

    spark.sql(MERGE_EXPIRE).show()

    spark.sql(INSERT_NEW).show()


In [0]:
# Start streaming query
query = (
    silver_df.writeStream
        .foreachBatch(upsert_to_gold)
        .option(
            "checkpointLocation",
            _checkpoints
        )
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()

In [0]:
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/gold/products/")
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/gold/products/", True)

In [0]:
%sql
-- make silver cleaned table CDF enabled

-- ALTER TABLE retails.silver.products_cleaned
-- SET TBLPROPERTIES (
--     delta.enableChangeDataFeed = true
-- )

-- DESCRIBE TABLE EXTENDED retails.silver.products_cleaned;